# Daniel OS grounded SFT: Colab GPU experiment

This notebook audits the old loss curve, generates diverse prompts while keeping every answer curated, builds scenario-family-disjoint train/validation splits, and compares LFM2 with LFM2.5 across three learning rates. Publication remains disabled until the frozen strict-test baseline is met or exceeded.

The data design follows the quality-over-volume findings in [LIMA](https://papers.neurips.cc/paper_files/paper/2023/hash/ac662d74829e4407ce1d126477f4a03a-Abstract-Conference.html), the generate-and-filter pattern in [Self-Instruct](https://aclanthology.org/2023.acl-long.754/), and the quality/complexity/diversity framework in [DEITA](https://proceedings.iclr.cc/paper_files/paper/2024/hash/6091f2bb355e960600f62566ac0e2862-Abstract-Conference.html).

In [ ]:
import os, pathlib, subprocess, sys

REPO = pathlib.Path('/content/sangbumchoi.github.io')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/SangbumChoi/sangbumchoi.github.io.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch>=2.6', 'transformers>=4.55,<5', 'trl>=0.24,<0.30', 'peft>=0.17,<1', 'datasets>=3,<5', 'accelerate>=1.2,<2', 'bitsandbytes>=0.45,<1', 'huggingface-hub>=0.34,<2', 'matplotlib>=3.9'], check=True)
print(subprocess.run(['nvidia-smi'], text=True, capture_output=True, check=True).stdout)

## 1. Diagnose the previous run

The old loss holdout had only two records per behavior. The audit also reconstructs how much cyclic oversampling occurred and checks the frozen evaluations for exact or near prompt overlap.

In [ ]:
subprocess.run([sys.executable, 'scripts/analyze_daniel_lfm2_data.py'], check=True)
diagnostics = __import__('json').loads(pathlib.Path('artifacts/daniel-lfm2-data-diagnostics.json').read_text())
print(__import__('json').dumps({
    'legacy_sampling': diagnostics['legacy_sampling'],
    'loss': diagnostics['loss'],
    'near_prompt_matches': diagnostics['prompt_overlap']['near_prompt_matches'],
}, indent=2))

## 2. Refresh the source survey

This verifies source availability, hashes retrieved pages, and creates review excerpts. Network text is never promoted directly into SFT. A human must update the curated profile before a new fact can become a target.

In [ ]:
subprocess.run([sys.executable, 'scripts/survey_daniel_public_sources.py'], check=True)
survey = __import__('json').loads(pathlib.Path('artifacts/daniel-public-source-survey.json').read_text())
survey['summary']

## 3. Generate grounded prompt diversity

A 4-bit Qwen3-4B teacher writes prompts only. The canonical answer, context keys, evidence, behavior, and language come from curated seeds. Entity definitions are expanded from the cited entity index, and neutral world-knowledge questions teach a search tool call rather than model-memory answers.

In [ ]:
subprocess.run([
    sys.executable, 'scripts/generate_daniel_lfm2_synthetic.py',
    '--output-dir', 'artifacts/daniel-lfm2-v3',
    '--batch-size', '4',
    '--temperature', '0.7',
], check=True)
subprocess.run([
    sys.executable, 'scripts/validate_daniel_lfm2_data.py',
    '--prepared-train', 'artifacts/daniel-lfm2-v3/train.jsonl',
    '--prepared-validation', 'artifacts/daniel-lfm2-v3/validation.jsonl',
], check=True)

## 4. GPU ablation sweep

The sweep uses an equal-size per-behavior validation pool for checkpoint selection and also logs full per-behavior loss. Evaluation runs every 25 optimizer steps, with two-evaluation early stopping. The frozen behavioral and strict suites are run for every candidate.

In [ ]:
import gc, json, torch
gc.collect(); torch.cuda.empty_cache()

models = ['LiquidAI/LFM2-350M', 'LiquidAI/LFM2.5-350M']
learning_rates = [5e-5, 1e-4, 2e-4]
sweep_root = pathlib.Path('artifacts/daniel-lfm2-sweep')
sweep_root.mkdir(parents=True, exist_ok=True)

for model in models:
    for lr in learning_rates:
        run_name = f"{model.split('/')[-1]}_lr{lr:.0e}"
        output = sweep_root / run_name
        command = [
            sys.executable, 'scripts/train_daniel_lfm2.py',
            '--model', model,
            '--prepared-train', 'artifacts/daniel-lfm2-v3/train.jsonl',
            '--prepared-validation', 'artifacts/daniel-lfm2-v3/validation.jsonl',
            '--output', str(output),
            '--epochs', '3', '--max-length', '1152',
            '--batch-size', '8', '--eval-batch-size', '8',
            '--gradient-accumulation-steps', '4',
            '--learning-rate', str(lr), '--weight-decay', '0.01',
            '--eval-steps', '25', '--early-stopping-patience', '2',
            '--training-revision', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
        ]
        with (output.parent / f'{run_name}.log').open('w') as log:
            subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, check=True)
        subprocess.run([
            sys.executable, 'scripts/evaluate_daniel_lfm2_test.py',
            '--model', str(output / 'merged'),
            '--output', str(output / 'strict-evaluation.json'),
            '--minimum-strict-score', '0.0', '--minimum-answer-score', '0.0',
            '--minimum-retrieve-score', '0.0', '--minimum-unknown-score', '0.0',
            '--minimum-refuse-score', '0.0', '--minimum-korean-score', '0.0',
        ], check=True)
        gc.collect(); torch.cuda.empty_cache()
        print('finished', run_name)

## 5. Select, plot, and export the result

A run is publishable only if strict, macro, answer, unknown, retrieval, refusal, hallucination, and Korean gates are all at least as strong as the currently deployed checkpoint.

In [ ]:
subprocess.run([sys.executable, 'scripts/select_daniel_lfm2_candidate.py', str(sweep_root)], check=True)
selection = json.loads((sweep_root / 'selection.json').read_text())
print(json.dumps(selection, indent=2))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 5))
for run in selection['runs']:
    metrics = json.loads((pathlib.Path(run['path']) / 'training_metrics.json').read_text())
    points = [p for p in metrics['validation'] if p.get('dataset') == 'macro' and p.get('loss') is not None]
    ax.plot([p['step'] for p in points], [p['loss'] for p in points], marker='o', label=run['run'])
ax.set(xlabel='optimizer step', ylabel='balanced validation loss', title='Daniel OS LFM2 ablation')
ax.grid(alpha=.25); ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(sweep_root / 'validation-loss-ablation.png', dpi=180)
plt.show()

In [ ]:
# Deliberate release switch. Keep False until the selection report is reviewed.
PUBLISH = False
if PUBLISH:
    assert selection['publication_allowed'], 'No candidate passed every frozen baseline gate.'
    from google.colab import userdata
    from huggingface_hub import HfApi
    token = userdata.get('HF_TOKEN')
    assert token, 'Add HF_TOKEN to Colab Secrets; never paste it into the notebook.'
    selected = pathlib.Path(selection['selected']['path']) / 'merged'
    api = HfApi(token=token)
    api.create_repo('danelcsb/daniel-lfm2-350m-candidate', repo_type='model', exist_ok=True)
    api.upload_folder(
        repo_id='danelcsb/daniel-lfm2-350m-candidate',
        repo_type='model', folder_path=selected,
        commit_message='Upload strict-gated Daniel OS candidate',
    )
    print('Candidate uploaded. Browser ONNX remains unchanged until its export and smoke tests pass.')